In [3]:
import pandas as pd
import requests
from io import StringIO

url = "https://moneypuck.com/moneypuck/playerData/careers/gameByGame/all_teams.csv"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

df = pd.read_csv(StringIO(response.text))

print(df.head())

  team  season name      gameId playerTeam opposingTeam home_or_away  \
0  NYR    2008  NYR  2008020001        NYR          T.B         AWAY   
1  NYR    2008  NYR  2008020001        NYR          T.B         AWAY   
2  NYR    2008  NYR  2008020001        NYR          T.B         AWAY   
3  NYR    2008  NYR  2008020001        NYR          T.B         AWAY   
4  NYR    2008  NYR  2008020001        NYR          T.B         AWAY   

   gameDate    position situation  ...  unblockedShotAttemptsAgainst  \
0  20081004  Team Level     other  ...                           1.0   
1  20081004  Team Level       all  ...                          31.0   
2  20081004  Team Level      5on5  ...                          20.0   
3  20081004  Team Level      4on5  ...                           9.0   
4  20081004  Team Level      5on4  ...                           1.0   

   scoreAdjustedUnblockedShotAttemptsAgainst  dZoneGiveawaysAgainst  \
0                                      1.000                   

In [4]:
# necháme jen celkové statistiky za zápas a jen regular season
nhl = df[df["situation"] == "all"].copy()
nhl = nhl[nhl["playoffGame"] == 0].copy()

nhl.shape

(43216, 111)

In [5]:
faceoff_data = nhl[[
    "season",
    "gameId",
    "gameDate",
    "team",
    "opposingTeam",
    "home_or_away",
    "goalsFor",
    "goalsAgainst",
    "faceOffsWonFor",
    "faceOffsWonAgainst"
]].copy()

faceoff_data.head()

,season,gameId,gameDate,team,opposingTeam,home_or_away,goalsFor,goalsAgainst,faceOffsWonFor,faceOffsWonAgainst
1,2008,2008020001,20081004,NYR,T.B,AWAY,2.0,1.0,30.0,32.0
6,2008,2008020003,20081005,NYR,T.B,HOME,2.0,1.0,17.0,29.0
11,2008,2008020010,20081010,NYR,CHI,HOME,4.0,2.0,23.0,32.0
16,2008,2008020019,20081011,NYR,PHI,AWAY,4.0,3.0,39.0,29.0
21,2008,2008020034,20081013,NYR,N.J,HOME,4.0,1.0,28.0,20.0


In [6]:
faceoff_data.groupby("gameId").size().value_counts()

2    21608
Name: count, dtype: int64

In [7]:
faceoff_data["total_faceoffs"] = (
    faceoff_data["faceOffsWonFor"] + faceoff_data["faceOffsWonAgainst"]
)

faceoff_data["faceoff_pct"] = (
    faceoff_data["faceOffsWonFor"] / faceoff_data["total_faceoffs"]
)

faceoff_data["faceoff_pct_100"] = faceoff_data["faceoff_pct"] * 100

faceoff_data.head()

,season,gameId,gameDate,team,opposingTeam,home_or_away,goalsFor,goalsAgainst,faceOffsWonFor,faceOffsWonAgainst,total_faceoffs,faceoff_pct,faceoff_pct_100
1,2008,2008020001,20081004,NYR,T.B,AWAY,2.0,1.0,30.0,32.0,62.0,0.483871,48.387097
6,2008,2008020003,20081005,NYR,T.B,HOME,2.0,1.0,17.0,29.0,46.0,0.369565,36.956522
11,2008,2008020010,20081010,NYR,CHI,HOME,4.0,2.0,23.0,32.0,55.0,0.418182,41.818182
16,2008,2008020019,20081011,NYR,PHI,AWAY,4.0,3.0,39.0,29.0,68.0,0.573529,57.352941
21,2008,2008020034,20081013,NYR,N.J,HOME,4.0,1.0,28.0,20.0,48.0,0.583333,58.333333


In [8]:
faceoff_data["win"] = (faceoff_data["goalsFor"] > faceoff_data["goalsAgainst"]).astype(int)

faceoff_data["goal_diff"] = faceoff_data["goalsFor"] - faceoff_data["goalsAgainst"]

faceoff_data.head()

,season,gameId,gameDate,team,opposingTeam,home_or_away,goalsFor,goalsAgainst,faceOffsWonFor,faceOffsWonAgainst,total_faceoffs,faceoff_pct,faceoff_pct_100,win,goal_diff
1,2008,2008020001,20081004,NYR,T.B,AWAY,2.0,1.0,30.0,32.0,62.0,0.483871,48.387097,1,1.0
6,2008,2008020003,20081005,NYR,T.B,HOME,2.0,1.0,17.0,29.0,46.0,0.369565,36.956522,1,1.0
11,2008,2008020010,20081010,NYR,CHI,HOME,4.0,2.0,23.0,32.0,55.0,0.418182,41.818182,1,2.0
16,2008,2008020019,20081011,NYR,PHI,AWAY,4.0,3.0,39.0,29.0,68.0,0.573529,57.352941,1,1.0
21,2008,2008020034,20081013,NYR,N.J,HOME,4.0,1.0,28.0,20.0,48.0,0.583333,58.333333,1,3.0


In [9]:
faceoff_data[[
    "team",
    "opposingTeam",
    "goalsFor",
    "goalsAgainst",
    "win",
    "goal_diff",
    "faceOffsWonFor",
    "faceOffsWonAgainst",
    "faceoff_pct_100"
]].head(10)

,team,opposingTeam,goalsFor,goalsAgainst,win,goal_diff,faceOffsWonFor,faceOffsWonAgainst,faceoff_pct_100
1,NYR,T.B,2.0,1.0,1,1.0,30.0,32.0,48.387097
6,NYR,T.B,2.0,1.0,1,1.0,17.0,29.0,36.956522
11,NYR,CHI,4.0,2.0,1,2.0,23.0,32.0,41.818182
16,NYR,PHI,4.0,3.0,1,1.0,39.0,29.0,57.352941
21,NYR,N.J,4.0,1.0,1,3.0,28.0,20.0,58.333333
26,NYR,BUF,1.0,3.0,0,-2.0,27.0,23.0,54.000000
31,NYR,TOR,0.0,0.0,0,0.0,24.0,33.0,42.105263
36,NYR,DET,4.0,5.0,0,-1.0,29.0,27.0,51.785714
41,NYR,DAL,1.0,2.0,0,-1.0,23.0,18.0,56.097561
46,NYR,CBJ,3.0,1.0,1,2.0,23.0,28.0,45.098039


In [10]:
faceoff_data = nhl[[
    "season",
    "gameId",
    "gameDate",
    "team",
    "opposingTeam",
    "home_or_away",
    "goalsFor",
    "goalsAgainst",
    "faceOffsWonFor",
    "faceOffsWonAgainst"
]].copy()

faceoff_data["total_faceoffs"] = (
    faceoff_data["faceOffsWonFor"] + faceoff_data["faceOffsWonAgainst"]
)

faceoff_data["faceoff_pct"] = (
    faceoff_data["faceOffsWonFor"] / faceoff_data["total_faceoffs"]
)

faceoff_data["faceoff_pct_100"] = faceoff_data["faceoff_pct"] * 100

faceoff_data.shape

(43216, 13)

In [33]:
faceoff_data.groupby("gameId").size().value_counts()

2    21608
Name: count, dtype: int64

In [12]:
faceoff_check = (
    faceoff_data
    .groupby("gameId")["faceoff_pct_100"]
    .sum()
    .reset_index(name="sum_faceoff_pct")
)

faceoff_check["sum_faceoff_pct"].describe()

count    2.160800e+04
mean     1.000000e+02
std      1.187986e-15
min      1.000000e+02
25%      1.000000e+02
50%      1.000000e+02
75%      1.000000e+02
max      1.000000e+02
Name: sum_faceoff_pct, dtype: float64

In [13]:
bad_faceoff_games = faceoff_check[
    abs(faceoff_check["sum_faceoff_pct"] - 100) > 0.000001
]

bad_faceoff_games

,gameId,sum_faceoff_pct


In [34]:
faceoff_data[faceoff_data["total_faceoffs"] == 0]

,season,gameId,gameDate,team,opposingTeam,home_or_away,goalsFor,goalsAgainst,faceOffsWonFor,faceOffsWonAgainst,total_faceoffs,faceoff_pct,faceoff_pct_100


In [35]:
faceoff_data["faceoff_pct_100"].describe()

count    43216.000000
mean        50.000000
std          7.394279
min         17.021277
25%         45.000000
50%         50.000000
75%         55.000000
max         82.978723
Name: faceoff_pct_100, dtype: float64

In [16]:
faceoff_situations = df[
    (df["situation"].isin(["5on5", "5on4", "4on5"])) &
    (df["playoffGame"] == 0)
].copy()

faceoff_situations = faceoff_situations[[
    "season",
    "gameId",
    "gameDate",
    "team",
    "opposingTeam",
    "home_or_away",
    "situation",
    "goalsFor",
    "goalsAgainst",
    "faceOffsWonFor",
    "faceOffsWonAgainst"
]].copy()

faceoff_situations.head(15)

,season,gameId,gameDate,team,opposingTeam,home_or_away,situation,goalsFor,goalsAgainst,faceOffsWonFor,faceOffsWonAgainst
2,2008,2008020001,20081004,NYR,T.B,AWAY,5on5,1.0,1.0,19.0,18.0
3,2008,2008020001,20081004,NYR,T.B,AWAY,4on5,0.0,0.0,5.0,2.0
4,2008,2008020001,20081004,NYR,T.B,AWAY,5on4,1.0,0.0,6.0,11.0
7,2008,2008020003,20081005,NYR,T.B,HOME,5on5,1.0,1.0,10.0,21.0
8,2008,2008020003,20081005,NYR,T.B,HOME,4on5,0.0,0.0,2.0,5.0
9,2008,2008020003,20081005,NYR,T.B,HOME,5on4,1.0,0.0,3.0,3.0
12,2008,2008020010,20081010,NYR,CHI,HOME,5on5,4.0,2.0,17.0,23.0
13,2008,2008020010,20081010,NYR,CHI,HOME,4on5,0.0,0.0,3.0,6.0
14,2008,2008020010,20081010,NYR,CHI,HOME,5on4,0.0,0.0,3.0,3.0
17,2008,2008020019,20081011,NYR,PHI,AWAY,5on5,3.0,2.0,20.0,21.0


In [17]:
faceoff_situations["total_faceoffs"] = (
    faceoff_situations["faceOffsWonFor"] + 
    faceoff_situations["faceOffsWonAgainst"]
)

faceoff_situations["faceoff_pct"] = (
    faceoff_situations["faceOffsWonFor"] / 
    faceoff_situations["total_faceoffs"]
)

faceoff_situations["faceoff_pct_100"] = (
    faceoff_situations["faceoff_pct"] * 100
)

faceoff_situations.head(15)

,season,gameId,gameDate,team,opposingTeam,home_or_away,situation,goalsFor,goalsAgainst,faceOffsWonFor,faceOffsWonAgainst,total_faceoffs,faceoff_pct,faceoff_pct_100
2,2008,2008020001,20081004,NYR,T.B,AWAY,5on5,1.0,1.0,19.0,18.0,37.0,0.513514,51.351351
3,2008,2008020001,20081004,NYR,T.B,AWAY,4on5,0.0,0.0,5.0,2.0,7.0,0.714286,71.428571
4,2008,2008020001,20081004,NYR,T.B,AWAY,5on4,1.0,0.0,6.0,11.0,17.0,0.352941,35.294118
7,2008,2008020003,20081005,NYR,T.B,HOME,5on5,1.0,1.0,10.0,21.0,31.0,0.322581,32.258065
8,2008,2008020003,20081005,NYR,T.B,HOME,4on5,0.0,0.0,2.0,5.0,7.0,0.285714,28.571429
9,2008,2008020003,20081005,NYR,T.B,HOME,5on4,1.0,0.0,3.0,3.0,6.0,0.500000,50.000000
12,2008,2008020010,20081010,NYR,CHI,HOME,5on5,4.0,2.0,17.0,23.0,40.0,0.425000,42.500000
13,2008,2008020010,20081010,NYR,CHI,HOME,4on5,0.0,0.0,3.0,6.0,9.0,0.333333,33.333333
14,2008,2008020010,20081010,NYR,CHI,HOME,5on4,0.0,0.0,3.0,3.0,6.0,0.500000,50.000000
17,2008,2008020019,20081011,NYR,PHI,AWAY,5on5,3.0,2.0,20.0,21.0,41.0,0.487805,48.780488


In [18]:
faceoff_situations[faceoff_situations["total_faceoffs"] == 0].shape

(2060, 14)

In [37]:
faceoff_situations[
    faceoff_situations["total_faceoffs"] == 0
]["situation"].value_counts()

situation
4on5    1030
5on4    1030
Name: count, dtype: int64

In [38]:
import numpy as np

faceoff_situations["faceoff_pct"] = np.where(
    faceoff_situations["total_faceoffs"] > 0,
    faceoff_situations["faceOffsWonFor"] / faceoff_situations["total_faceoffs"],
    np.nan
)

faceoff_situations["faceoff_pct_100"] = faceoff_situations["faceoff_pct"] * 100

faceoff_situations[[
    "season",
    "gameId",
    "team",
    "opposingTeam",
    "situation",
    "faceOffsWonFor",
    "faceOffsWonAgainst",
    "total_faceoffs",
    "faceoff_pct_100"
]].head(20)

,season,gameId,team,opposingTeam,situation,faceOffsWonFor,faceOffsWonAgainst,total_faceoffs,faceoff_pct_100
2,2008,2008020001,NYR,T.B,5on5,19.0,18.0,37.0,51.351351
3,2008,2008020001,NYR,T.B,4on5,5.0,2.0,7.0,71.428571
4,2008,2008020001,NYR,T.B,5on4,6.0,11.0,17.0,35.294118
7,2008,2008020003,NYR,T.B,5on5,10.0,21.0,31.0,32.258065
8,2008,2008020003,NYR,T.B,4on5,2.0,5.0,7.0,28.571429
9,2008,2008020003,NYR,T.B,5on4,3.0,3.0,6.0,50.000000
12,2008,2008020010,NYR,CHI,5on5,17.0,23.0,40.0,42.500000
13,2008,2008020010,NYR,CHI,4on5,3.0,6.0,9.0,33.333333
14,2008,2008020010,NYR,CHI,5on4,3.0,3.0,6.0,50.000000
17,2008,2008020019,NYR,PHI,5on5,20.0,21.0,41.0,48.780488


In [39]:
summary_by_situation = (
    faceoff_situations
    .groupby("situation")
    .agg(
        rows=("gameId", "count"),
        games=("gameId", "nunique"),
        zero_faceoff_rows=("total_faceoffs", lambda x: (x == 0).sum()),
        avg_total_faceoffs=("total_faceoffs", "mean"),
        median_total_faceoffs=("total_faceoffs", "median"),
        avg_faceoff_pct=("faceoff_pct_100", "mean"),
        median_faceoff_pct=("faceoff_pct_100", "median")
    )
    .reset_index()
)

summary_by_situation

,situation,rows,games,zero_faceoff_rows,avg_total_faceoffs,median_total_faceoffs,avg_faceoff_pct,median_faceoff_pct
0,4on5,43216,21608,1030,5.321617,5.0,46.387606,50.0
1,5on4,43216,21608,1030,5.321617,5.0,53.612394,50.0
2,5on5,43216,21608,0,44.143836,44.0,50.000000,50.0


In [22]:
situation_check = (
    faceoff_situations
    .groupby(["gameId", "situation"])["total_faceoffs"]
    .sum()
    .unstack()
)

situation_check.head()

situation,4on5,5on4,5on5
gameId,,,
2008020001,24.0,24.0,74.0
2008020002,19.0,19.0,62.0
2008020003,13.0,13.0,62.0
2008020004,23.0,23.0,68.0
2008020005,9.0,9.0,96.0


In [23]:
[col for col in df.columns if "shot" in col.lower()]

['shotsOnGoalFor',
 'missedShotsFor',
 'blockedShotAttemptsFor',
 'shotAttemptsFor',
 'savedShotsOnGoalFor',
 'savedUnblockedShotAttemptsFor',
 'lowDangerShotsFor',
 'mediumDangerShotsFor',
 'highDangerShotsFor',
 'scoreAdjustedShotsAttemptsFor',
 'unblockedShotAttemptsFor',
 'scoreAdjustedUnblockedShotAttemptsFor',
 'xGoalsFromxReboundsOfShotsFor',
 'xGoalsFromActualReboundsOfShotsFor',
 'totalShotCreditFor',
 'scoreAdjustedTotalShotCreditFor',
 'scoreFlurryAdjustedTotalShotCreditFor',
 'shotsOnGoalAgainst',
 'missedShotsAgainst',
 'blockedShotAttemptsAgainst',
 'shotAttemptsAgainst',
 'savedShotsOnGoalAgainst',
 'savedUnblockedShotAttemptsAgainst',
 'lowDangerShotsAgainst',
 'mediumDangerShotsAgainst',
 'highDangerShotsAgainst',
 'scoreAdjustedShotsAttemptsAgainst',
 'unblockedShotAttemptsAgainst',
 'scoreAdjustedUnblockedShotAttemptsAgainst',
 'xGoalsFromxReboundsOfShotsAgainst',
 'xGoalsFromActualReboundsOfShotsAgainst',
 'totalShotCreditAgainst',
 'scoreAdjustedTotalShotCreditAgai

In [40]:
shots_data = nhl[[
    "season",
    "gameId",
    "gameDate",
    "team",
    "opposingTeam",
    "home_or_away",
    "goalsFor",
    "goalsAgainst",
    "shotsOnGoalFor",
    "shotsOnGoalAgainst",
    "shotAttemptsFor",
    "shotAttemptsAgainst",
    "highDangerShotsFor",
    "highDangerShotsAgainst"
]].copy()

shots_data.head()

,season,gameId,gameDate,team,opposingTeam,home_or_away,goalsFor,goalsAgainst,shotsOnGoalFor,shotsOnGoalAgainst,shotAttemptsFor,shotAttemptsAgainst,highDangerShotsFor,highDangerShotsAgainst
1,2008,2008020001,20081004,NYR,T.B,AWAY,2.0,1.0,41.0,21.0,66.0,37.0,1.0,3.0
6,2008,2008020003,20081005,NYR,T.B,HOME,2.0,1.0,39.0,19.0,72.0,44.0,0.0,0.0
11,2008,2008020010,20081010,NYR,CHI,HOME,4.0,2.0,29.0,32.0,51.0,53.0,0.0,2.0
16,2008,2008020019,20081011,NYR,PHI,AWAY,4.0,3.0,27.0,28.0,50.0,50.0,5.0,3.0
21,2008,2008020034,20081013,NYR,N.J,HOME,4.0,1.0,24.0,27.0,45.0,58.0,2.0,6.0


In [25]:
game_situation_data = df[
    (df["situation"].isin(["5on5", "5on4", "4on5"])) &
    (df["playoffGame"] == 0)
].copy()

game_situation_data = game_situation_data[[
    "season",
    "gameId",
    "gameDate",
    "team",
    "opposingTeam",
    "home_or_away",
    "situation",
    
    # result in that situation
    "goalsFor",
    "goalsAgainst",
    
    # faceoffs
    "faceOffsWonFor",
    "faceOffsWonAgainst",
    
    # shots
    "shotsOnGoalFor",
    "shotsOnGoalAgainst",
    "shotAttemptsFor",
    "shotAttemptsAgainst",
    
    # giveaways
    "giveawaysFor",
    "giveawaysAgainst"
]].copy()

game_situation_data.head(15)

,season,gameId,gameDate,team,opposingTeam,home_or_away,situation,goalsFor,goalsAgainst,faceOffsWonFor,faceOffsWonAgainst,shotsOnGoalFor,shotsOnGoalAgainst,shotAttemptsFor,shotAttemptsAgainst,giveawaysFor,giveawaysAgainst
2,2008,2008020001,20081004,NYR,T.B,AWAY,5on5,1.0,1.0,19.0,18.0,28.0,11.0,45.0,25.0,7.0,5.0
3,2008,2008020001,20081004,NYR,T.B,AWAY,4on5,0.0,0.0,5.0,2.0,1.0,8.0,1.0,10.0,0.0,1.0
4,2008,2008020001,20081004,NYR,T.B,AWAY,5on4,1.0,0.0,6.0,11.0,12.0,1.0,20.0,1.0,1.0,1.0
7,2008,2008020003,20081005,NYR,T.B,HOME,5on5,1.0,1.0,10.0,21.0,26.0,13.0,51.0,32.0,10.0,6.0
8,2008,2008020003,20081005,NYR,T.B,HOME,4on5,0.0,0.0,2.0,5.0,3.0,4.0,4.0,7.0,1.0,0.0
9,2008,2008020003,20081005,NYR,T.B,HOME,5on4,1.0,0.0,3.0,3.0,6.0,1.0,12.0,2.0,0.0,1.0
12,2008,2008020010,20081010,NYR,CHI,HOME,5on5,4.0,2.0,17.0,23.0,22.0,26.0,40.0,45.0,5.0,4.0
13,2008,2008020010,20081010,NYR,CHI,HOME,4on5,0.0,0.0,3.0,6.0,2.0,2.0,4.0,4.0,0.0,1.0
14,2008,2008020010,20081010,NYR,CHI,HOME,5on4,0.0,0.0,3.0,3.0,5.0,2.0,7.0,2.0,1.0,0.0
17,2008,2008020019,20081011,NYR,PHI,AWAY,5on5,3.0,2.0,20.0,21.0,18.0,20.0,30.0,35.0,1.0,6.0


In [26]:
df["season"].min(), df["season"].max()

(np.int64(2008), np.int64(2025))

In [27]:

final_game_situation_data = df[
    (df["season"] < 2025) &
    (df["playoffGame"] == 0) &
    (df["situation"].isin(["5on5", "5on4", "4on5"]))
].copy()

final_game_situation_data = final_game_situation_data[[
    "season",
    "gameId",
    "gameDate",
    "team",
    "opposingTeam",
    "home_or_away",
    "situation",
    
    # goals in given situation
    "goalsFor",
    "goalsAgainst",
    
    # faceoffs
    "faceOffsWonFor",
    "faceOffsWonAgainst",
    
    # shots
    "shotsOnGoalFor",
    "shotsOnGoalAgainst",
    "shotAttemptsFor",
    "shotAttemptsAgainst",
    
    # giveaways
    "giveawaysFor",
    "giveawaysAgainst"
]].copy()

final_game_situation_data["win"] = (
    final_game_situation_data["goalsFor"] > final_game_situation_data["goalsAgainst"]
).astype(int)

final_game_situation_data["goal_diff"] = (
    final_game_situation_data["goalsFor"] - final_game_situation_data["goalsAgainst"]
)

final_game_situation_data.head(15)

,season,gameId,gameDate,team,opposingTeam,home_or_away,situation,goalsFor,goalsAgainst,faceOffsWonFor,faceOffsWonAgainst,shotsOnGoalFor,shotsOnGoalAgainst,shotAttemptsFor,shotAttemptsAgainst,giveawaysFor,giveawaysAgainst,win,goal_diff
2,2008,2008020001,20081004,NYR,T.B,AWAY,5on5,1.0,1.0,19.0,18.0,28.0,11.0,45.0,25.0,7.0,5.0,0,0.0
3,2008,2008020001,20081004,NYR,T.B,AWAY,4on5,0.0,0.0,5.0,2.0,1.0,8.0,1.0,10.0,0.0,1.0,0,0.0
4,2008,2008020001,20081004,NYR,T.B,AWAY,5on4,1.0,0.0,6.0,11.0,12.0,1.0,20.0,1.0,1.0,1.0,1,1.0
7,2008,2008020003,20081005,NYR,T.B,HOME,5on5,1.0,1.0,10.0,21.0,26.0,13.0,51.0,32.0,10.0,6.0,0,0.0
8,2008,2008020003,20081005,NYR,T.B,HOME,4on5,0.0,0.0,2.0,5.0,3.0,4.0,4.0,7.0,1.0,0.0,0,0.0
9,2008,2008020003,20081005,NYR,T.B,HOME,5on4,1.0,0.0,3.0,3.0,6.0,1.0,12.0,2.0,0.0,1.0,1,1.0
12,2008,2008020010,20081010,NYR,CHI,HOME,5on5,4.0,2.0,17.0,23.0,22.0,26.0,40.0,45.0,5.0,4.0,1,2.0
13,2008,2008020010,20081010,NYR,CHI,HOME,4on5,0.0,0.0,3.0,6.0,2.0,2.0,4.0,4.0,0.0,1.0,0,0.0
14,2008,2008020010,20081010,NYR,CHI,HOME,5on4,0.0,0.0,3.0,3.0,5.0,2.0,7.0,2.0,1.0,0.0,0,0.0
17,2008,2008020019,20081011,NYR,PHI,AWAY,5on5,3.0,2.0,20.0,21.0,18.0,20.0,30.0,35.0,1.0,6.0,1,1.0


In [28]:
final_game_situation_data.sort_values(
    ["season", "gameDate", "gameId", "team", "situation"]
).tail(20)

,season,gameId,gameDate,team,opposingTeam,home_or_away,situation,goalsFor,goalsAgainst,faceOffsWonFor,faceOffsWonAgainst,shotsOnGoalFor,shotsOnGoalAgainst,shotAttemptsFor,shotAttemptsAgainst,giveawaysFor,giveawaysAgainst,win,goal_diff
146829,2024,2024021309,20250417,OTT,CAR,HOME,5on4,2.0,0.0,6.0,3.0,5.0,0.0,11.0,0.0,0.0,1.0,1,2.0
146827,2024,2024021309,20250417,OTT,CAR,HOME,5on5,4.0,4.0,20.0,25.0,22.0,24.0,43.0,45.0,10.0,13.0,0,0.0
6608,2024,2024021310,20250417,NYR,TBL,HOME,4on5,1.0,0.0,8.0,0.0,2.0,0.0,3.0,2.0,2.0,0.0,1,1.0
6609,2024,2024021310,20250417,NYR,TBL,HOME,5on4,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0
6607,2024,2024021310,20250417,NYR,TBL,HOME,5on5,3.0,0.0,21.0,22.0,20.0,27.0,43.0,47.0,13.0,3.0,1,3.0
159088,2024,2024021310,20250417,TBL,NYR,AWAY,4on5,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0
159089,2024,2024021310,20250417,TBL,NYR,AWAY,5on4,0.0,1.0,0.0,8.0,0.0,2.0,2.0,3.0,0.0,2.0,0,-1.0
159087,2024,2024021310,20250417,TBL,NYR,AWAY,5on5,0.0,3.0,22.0,21.0,27.0,20.0,47.0,43.0,3.0,13.0,0,-3.0
16423,2024,2024021311,20250417,PIT,WSH,HOME,4on5,1.0,1.0,0.0,2.0,1.0,1.0,1.0,6.0,1.0,0.0,0,0.0
16424,2024,2024021311,20250417,PIT,WSH,HOME,5on4,1.0,0.0,2.0,2.0,4.0,1.0,8.0,1.0,0.0,0.0,1,1.0


In [29]:
summary_by_situation = (
    final_game_situation_data
    .groupby("situation")
    .agg(
        games=("gameId", "nunique"),
        rows=("gameId", "count"),
        
        avg_goals_for=("goalsFor", "mean"),
        avg_goals_against=("goalsAgainst", "mean"),
        
        avg_faceoffs_won_for=("faceOffsWonFor", "mean"),
        avg_faceoffs_won_against=("faceOffsWonAgainst", "mean"),
        
        avg_shots_on_goal_for=("shotsOnGoalFor", "mean"),
        avg_shots_on_goal_against=("shotsOnGoalAgainst", "mean"),
        
        avg_shot_attempts_for=("shotAttemptsFor", "mean"),
        avg_shot_attempts_against=("shotAttemptsAgainst", "mean"),
        
        avg_giveaways_for=("giveawaysFor", "mean"),
        avg_giveaways_against=("giveawaysAgainst", "mean")
    )
    .reset_index()
)

summary_by_situation

,situation,games,rows,avg_goals_for,avg_goals_against,avg_faceoffs_won_for,avg_faceoffs_won_against,avg_shots_on_goal_for,avg_shots_on_goal_against,avg_shot_attempts_for,avg_shot_attempts_against,avg_giveaways_for,avg_giveaways_against
0,4on5,20296,40592,0.071246,0.548680,2.483322,2.874581,0.802276,4.289688,1.166240,8.016481,0.499901,0.794516
1,5on4,20296,40592,0.548680,0.071246,2.874581,2.483322,4.289688,0.802276,8.016481,1.166240,0.794516,0.499901
2,5on5,20296,40592,1.901458,1.901458,22.098985,22.098985,23.491920,23.491920,44.272221,44.272221,7.385593,7.385593


In [30]:
[col for col in final_game_situation_data.columns if "win" in col.lower()]

['win']

In [31]:

game_results = df[
    (df["season"] < 2025) &
    (df["playoffGame"] == 0) &
    (df["situation"] == "all")
].copy()

game_results = game_results[[
    "gameId",
    "team",
    "goalsFor",
    "goalsAgainst"
]].copy()

game_results = game_results.rename(columns={
    "goalsFor": "game_goals_for",
    "goalsAgainst": "game_goals_against"
})

# win / draw / lose za celý zápas
game_results["win"] = 0
game_results["draw"] = 0
game_results["lose"] = 0

game_results.loc[
    game_results["game_goals_for"] > game_results["game_goals_against"],
    "win"
] = 1

game_results.loc[
    game_results["game_goals_for"] == game_results["game_goals_against"],
    "draw"
] = 1

game_results.loc[
    game_results["game_goals_for"] < game_results["game_goals_against"],
    "lose"
] = 1

# points: NHL tabulkové body
# výhra = 2 body, remíza = 1 bod, prohra = 0 bodů
game_results["points"] = (
    game_results["win"] * 2 +
    game_results["draw"] * 1
)

game_results["game_goal_diff"] = (
    game_results["game_goals_for"] - game_results["game_goals_against"]
)

game_results.head()

,gameId,team,game_goals_for,game_goals_against,win,draw,lose,points,game_goal_diff
1,2008020001,NYR,2.0,1.0,1,0,0,2,1.0
6,2008020003,NYR,2.0,1.0,1,0,0,2,1.0
11,2008020010,NYR,4.0,2.0,1,0,0,2,2.0
16,2008020019,NYR,4.0,3.0,1,0,0,2,1.0
21,2008020034,NYR,4.0,1.0,1,0,0,2,3.0


In [41]:

final_game_situation_data = final_game_situation_data.drop(
    columns=[
        "win", 
        "draw", 
        "lose", 
        "points",
        "game_goals_for",
        "game_goals_against",
        "game_goal_diff"
    ],
    errors="ignore"
)

# merge do hlavního working datasetu
final_game_situation_data = final_game_situation_data.merge(
    game_results,
    on=["gameId", "team"],
    how="left"
)

final_game_situation_data.tail(10)

,season,gameId,gameDate,team,opposingTeam,home_or_away,situation,goalsFor,goalsAgainst,faceOffsWonFor,...,giveawaysFor,giveawaysAgainst,goal_diff,game_goals_for,game_goals_against,win,draw,lose,points,game_goal_diff
121766,2020,2020020867,20210508,L.A,COL,HOME,5on4,0.0,0.0,1.0,...,0.0,1.0,0.0,2.0,3.0,0,0,1,0,-1.0
121767,2020,2020020456,20210510,L.A,STL,HOME,5on5,1.0,1.0,20.0,...,4.0,4.0,0.0,1.0,2.0,0,0,1,0,-1.0
121768,2020,2020020456,20210510,L.A,STL,HOME,4on5,0.0,0.0,0.0,...,2.0,0.0,0.0,1.0,2.0,0,0,1,0,-1.0
121769,2020,2020020456,20210510,L.A,STL,HOME,5on4,0.0,0.0,2.0,...,1.0,0.0,0.0,1.0,2.0,0,0,1,0,-1.0
121770,2020,2020020688,20210512,L.A,COL,AWAY,5on5,0.0,5.0,19.0,...,2.0,4.0,-5.0,0.0,6.0,0,0,1,0,-6.0
121771,2020,2020020688,20210512,L.A,COL,AWAY,4on5,0.0,1.0,3.0,...,0.0,0.0,-1.0,0.0,6.0,0,0,1,0,-6.0
121772,2020,2020020688,20210512,L.A,COL,AWAY,5on4,0.0,0.0,2.0,...,0.0,0.0,0.0,0.0,6.0,0,0,1,0,-6.0
121773,2020,2020020704,20210513,L.A,COL,AWAY,5on5,1.0,4.0,29.0,...,0.0,2.0,-3.0,1.0,5.0,0,0,1,0,-4.0
121774,2020,2020020704,20210513,L.A,COL,AWAY,4on5,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,5.0,0,0,1,0,-4.0
121775,2020,2020020704,20210513,L.A,COL,AWAY,5on4,0.0,0.0,2.0,...,0.0,0.0,0.0,1.0,5.0,0,0,1,0,-4.0
